# 第1章：数字图像的获取和表示

## 编程实践：手写 Gamma 校正

| 项目 | 说明 |
|------|------|
| 输入图片 | `lena.jpeg`（来自 Hands-on-CV 参考库第 2/5 章） |
| 手写核心 | 显式三重循环实现 Gamma 校正 |
| 允许调用 | 仅 `cv_imread` / `cv_imwrite` 图像读写 |
| 对比验证 | 与 OpenCV `cv2.LUT` 结果做数值误差对比 |


## 一、学习目标

1. 数字图像是相机拍摄三维物理世界得到的平面投影，能说清每个像素值的物理含义。
2. 数码相机成像的 **ISP 管线** 包含哪些环节，每个环节在做什么。
3. 数字图像以**像素矩阵**方式记录，理解该矩阵的特点（离散、有限、量化、空间相关）。
4. 理解 Gamma 校正的公式、参数作用与典型应用场景。

> 本资料为掌握基础知识的学习指南，应以拓展学习和扎实积累为目的。学习完成后需能现场给出代码演示并回答概念理解与算法实现相关问题。


## 二、原理：数字图像是什么

数字图像本质是一个 **二维矩阵**，每个元素称为 **像素（Pixel）**：

```
图像 = 一个 H 行 × W 列的像素矩阵
```

- **灰度图**：每个像素一个值，表示亮度，通常取 `0~255`（0 黑、255 白）。
- **彩色图**：每个像素三个值，OpenCV 使用 **BGR** 顺序，三个通道各 `0~255`。
- **ISP 管线**：光线 → 镜头 → 光圈 → 快门 → 感光器(CCD/CMOS) → ADC → ISP 处理 → 数字图像。
  ISP 中常见处理：去马赛克（demosaic）、白平衡、降噪、色彩校正、Gamma 校正、锐化等。
- **像素矩阵特点**：离散性、有限性、量化误差、相邻像素空间相关性。


## 三、原理：Gamma 校正

Gamma 校正是对像素值做**幂函数**的非线性变换：

```
I_out = 255 * (I_in / 255) ** (1 / gamma)
```

- `gamma > 1`：曲线下凹，暗部被拉伸，整体**变亮**。
- `gamma < 1`：曲线上凸，暗部被压缩，整体**变暗**。
- `gamma = 1`：恒等变换。
- `gamma = 2.2`：CRT 显示器的典型值，也是 sRGB 显示校正常用值。

用途：显示器校准、暗部增强、摄影后期、医学影像显示等。核心思想是补偿人眼对亮度的**对数式非线性感知**。


## 四、手写约束清单

> 老师要求：除 OpenCV 的图像读写函数外，其余代码全部自己手写，不能调用其他函数库。

| 类型 | 是否允许 | 说明 |
|------|:---:|------|
| `cv_imread` / `cv_imwrite` | ✅ | 图像读写 |
| Python 循环 / 条件 / 算术 | ✅ | 核心算法 |
| `np.zeros` 等数组分配 | ✅ | 仅用于开辟结果空间 |
| `cv2.LUT` / `cv2.cvtColor` | ❌ | 属于图像处理函数，实现中不得使用 |
| `np.power` / `np.vectorize` | ❌ | 属于向量化函数，实现中不得使用 |
| `matplotlib` | ✅ | 仅用于可视化，不参与算法计算 |
| 与 OpenCV 的对比验证 | ✅ | 仅放在"验证"单元格，不属于手写实现 |


In [ ]:
import sys
from pathlib import Path

# 向上查找项目根目录（含 utils.py），并加入 sys.path
ROOT = Path.cwd().resolve()
while not (ROOT / "utils.py").exists():
    if ROOT.parent == ROOT:
        raise FileNotFoundError("未找到项目根目录 utils.py")
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import cv2
import matplotlib.pyplot as plt

from utils import cv_imread, cv_imwrite, set_random_seed, setup_plot_chinese, show_images, compare_results

setup_plot_chinese()
set_random_seed(42)
print(f"OpenCV 版本: {cv2.__version__}")
print(f"当前工作目录: {Path.cwd()}")


In [ ]:
def gamma_correction_manual(image, gamma):
    """手写 Gamma 校正。

    参数
    ----
    image : np.ndarray
        输入 BGR 图像，形状 (H, W, 3)，dtype=uint8。
    gamma : float
        Gamma 值，大于 0。

    返回
    ----
    np.ndarray
        校正后的 uint8 图像。

    说明
    ----
    使用显式三重循环逐像素计算 I_out = 255 * (I_in / 255) ** (1 / gamma)，
    不调用 np.power / cv2.LUT 等向量化或库函数。
    """
    h, w, c = image.shape
    result = np.zeros((h, w, c), dtype=np.uint8)
    inv_gamma = 1.0 / gamma

    for y in range(h):
        for x in range(w):
            for ch in range(c):
                value = float(image[y, x, ch]) / 255.0
                corrected = value ** inv_gamma
                result[y, x, ch] = int(round(corrected * 255.0))
    return result

# 自测：3x3 小图像，gamma=1 时应保持原值不变
small = np.arange(27, dtype=np.uint8).reshape(3, 3, 3)
identity = gamma_correction_manual(small, 1.0)
assert np.array_equal(small, identity), "gamma=1 应保持原图不变"
print("自测通过：gamma=1 时输出与输入一致")


In [ ]:
# 读取输入图像并执行手写 Gamma 校正
img = cv_imread("lena.jpeg", cv2.IMREAD_COLOR)
assert img is not None, "读取 lena.jpeg 失败"

gamma = 2.2
out = gamma_correction_manual(img, gamma)

# ---------- 与 OpenCV 对比验证（仅验证，不用于实现） ----------
lut = np.array([(i / 255.0) ** (1.0 / gamma) * 255.0 for i in range(256)], dtype=np.uint8)
ref = cv2.LUT(img, lut)
compare_results(out, ref, "Gamma 校正")

cv_imwrite("gamma_corrected.jpg", out)
show_images([img, out, ref], [f"原图", f"手写 gamma={gamma}", "OpenCV LUT"], figsize=(13, 4))


## 五、结果与参数分析

- 与 `cv2.LUT` 的参考结果对比，误差应只在 `±1` 个灰度级以内（由 `int(round(...))` 取整造成），MAE 接近 0。
- 可以尝试 `gamma=0.5 / 1.0 / 3.0`，观察图像变暗 / 不变 / 变亮。
- 三重循环在大图上较慢，这正是"手写实现"与工程上向量化实现之间的权衡：教学中先吃透逐像素逻辑，再理解为何工程代码会用查表（LUT）加速。

**易错点**
1. OpenCV 通道顺序是 **BGR** 而非 RGB。
2. 必须先归一化到 `[0,1]` 再做幂运算，最后反归一化回 `[0,255]`。
3. 用 `int(round(...))` 而不是 `int(...)`，减小截断误差。
4. JPEG 是有损压缩，保存后再读取可能引入微小像素差异。


## 六、科研规范小结

1. **可复现**：所有随机操作固定种子；本页无随机量，但统一调用 `set_random_seed(42)` 作为约定。
2. **函数化**：核心逻辑封装为带 docstring 的函数，便于复用和测试。
3. **对比验证**：手写实现必须与可信的库实现做数值对比，量化误差而非"看起来差不多"。
4. **边界自测**：`gamma=1` 的恒等断言，用于快速发现实现错误。


## 七、练习：手写图像基本操作

**要求**（除 OpenCV 读写外全部手写）：
1. 手写灰度化（ITU-R BT.601 权重），不调用 `cv2.cvtColor`。
2. 手写二值化（阈值 127）。
3. 手写水平 / 垂直翻转。
4. 手写亮度调整（逐像素加减一个偏置并裁剪到 `[0,255]`）。
5. 与 OpenCV 对应函数做数值对比。


In [ ]:
# ==================== 练习解决方案 ====================
def to_grayscale_manual(image):
    """手写灰度化：Y = 0.299R + 0.587G + 0.114B。"""
    h, w, c = image.shape
    gray = np.zeros((h, w), dtype=np.uint8)
    for y in range(h):
        for x in range(w):
            b, g, r = (float(v) for v in image[y, x, :])
            gray[y, x] = int(round(0.299 * r + 0.587 * g + 0.114 * b))
    return gray


def threshold_manual(image, threshold=127):
    """手写二值化：大于阈值置 255，否则置 0。"""
    gray = to_grayscale_manual(image) if image.ndim == 3 else image
    h, w = gray.shape
    binary = np.zeros((h, w), dtype=np.uint8)
    for y in range(h):
        for x in range(w):
            binary[y, x] = 255 if gray[y, x] > threshold else 0
    return binary


def flip_manual(image, axis=0):
    """手写翻转：axis=0 垂直翻转，axis=1 水平翻转。"""
    h, w = image.shape[:2]
    if image.ndim == 2:
        result = np.zeros((h, w), dtype=image.dtype)
        for y in range(h):
            for x in range(w):
                result[y, x] = image[h - 1 - y, x] if axis == 0 else image[y, w - 1 - x]
    else:
        result = np.zeros_like(image)
        for y in range(h):
            for x in range(w):
                result[y, x, :] = image[h - 1 - y, x, :] if axis == 0 else image[y, w - 1 - x, :]
    return result


def adjust_brightness_manual(image, delta=40):
    """手写亮度调整：逐像素加 delta 并裁剪到 [0,255]。"""
    result = np.zeros_like(image)
    if image.ndim == 2:
        h, w = image.shape
        for y in range(h):
            for x in range(w):
                result[y, x] = min(255, max(0, int(image[y, x]) + delta))
    else:
        h, w, c = image.shape
        for y in range(h):
            for x in range(w):
                for ch in range(c):
                    result[y, x, ch] = min(255, max(0, int(image[y, x, ch]) + delta))
    return result


# 验证
img = cv_imread("lena.jpeg", cv2.IMREAD_COLOR)
gray_m = to_grayscale_manual(img)
gray_cv = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
compare_results(gray_m, gray_cv, "灰度化")

binary_m = threshold_manual(img, 127)
_, binary_cv = cv2.threshold(gray_m, 127, 255, cv2.THRESH_BINARY)
compare_results(binary_m, binary_cv, "二值化")

flip_m = flip_manual(img, 0)
flip_cv = cv2.flip(img, 0)
compare_results(flip_m, flip_cv, "垂直翻转")

bright_m = adjust_brightness_manual(img, 40)
bright_cv = cv2.convertScaleAbs(img, alpha=1.0, beta=40)
compare_results(bright_m, bright_cv, "亮度调整")

show_images([gray_m, binary_m, flip_m, bright_m], ["灰度化", "二值化", "垂直翻转", "亮度+40"], figsize=(13, 4))


### 代码要点解释

1. **灰度化**：使用 BT.601 标准权重 `0.299/0.587/0.114`，与 OpenCV 默认一致。
2. **二值化**：比较符用 `>`，与 `cv2.THRESH_BINARY` 一致。
3. **翻转**：通过坐标映射 `(h-1-y, x)` / `(y, w-1-x)` 实现，避免新建大数组之外的任何库调用。
4. **亮度调整**：逐像素加减并用 `min/max` 裁剪，与 `cv2.convertScaleAbs` 结果一致。
